In [ ]:
import pandas as pd
import numpy as np
import re

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error

In [ ]:
np.random.seed(42)
n = 1000

In [ ]:
sqft = np.random.randint(800, 5000, n)
bedrooms = np.random.randint(1, 6, n)

In [ ]:
noise = np.random.normal(0, 20000, n)
price = 50 * sqft + 30000 * bedrooms + 80000 + noise
price = price.astype(int)

In [ ]:
addresses = [f"{i} Main St, Portland, OR {97000 + (i % 100):05d}" for i in range(n)]

In [ ]:
df = pd.DataFrame({
    "price": price,
    "sqft": sqft,
    "bedrooms": bedrooms,
    "abbreviatedAddress": addresses
})

print("Shape:", df.shape)

In [ ]:
df.to_csv("/tmp/portland_housing.csv", index=False)
df = pd.read_csv("/tmp/portland_housing.csv", low_memory=False)
print("Dataset loaded successfully")

In [ ]:
print(df.head())

In [ ]:
print(df.dtypes)

In [ ]:
print(df.columns.tolist())

In [ ]:
print(df.info())

In [ ]:
print(df.describe())

In [ ]:
df['zip_extracted'] = df['abbreviatedAddress'].astype(str).apply(
    lambda x: re.findall(r'\d{5}', x)[0] if re.findall(r'\d{5}', x) else None
)
print("Columns:", df.columns.tolist())
print(df['zip_extracted'].head(10))

In [ ]:
df['clean_address'] = df['abbreviatedAddress'].astype(str).apply(
    lambda x: re.sub(r'[^a-zA-Z0-9 ]', '', x)
)
print(df['clean_address'].head())

In [ ]:
df['addr_number'] = df['abbreviatedAddress'].astype(str).apply(
    lambda x: re.findall(r'^\d+', x)
)
df['addr_number'] = df['addr_number'].apply(lambda x: x[0] if x else None)
print(df['addr_number'].head())

In [ ]:
df['has_direction'] = df['abbreviatedAddress'].astype(str).apply(
    lambda x: bool(re.search(r'\b(NE|NW|SE|SW)\b', x))
)
print(df['has_direction'].value_counts())

In [ ]:
print('Null counts:')
print(df.isnull().sum().sort_values(ascending=False).head(20))

In [ ]:
for col in df.select_dtypes(include=[np.number]).columns:
    if df[col].isnull().sum() > 0:
        df[col].fillna(df[col].median(), inplace=True)
print('Nulls after filling:', df.isnull().sum().sum())

In [ ]:
print(df.isnull().any())

In [ ]:
print('Duplicate rows:', df.duplicated().sum())
df.drop_duplicates(inplace=True)
print('After removing duplicates:', df.shape)

In [ ]:
df.columns = df.columns.str.strip().str.lower()
print(df.columns.tolist())

In [ ]:
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df[['price', 'sqft', 'bedrooms']].fillna(
    df[['price', 'sqft', 'bedrooms']].median(), inplace=True
)
print('Shape after formatting:', df.shape)

In [ ]:
bins = [0, 1000, 2000, 3000, 6000]
labels = ['Small', 'Medium', 'Large', 'Very Large']
df['area_bin'] = pd.cut(df['sqft'], bins=bins, labels=labels)
print(df['area_bin'].value_counts())

In [ ]:
price_bins = [0, 150000, 300000, 500000, 1000000]
price_labels = ['Budget', 'Mid', 'Upper-Mid', 'Luxury']
df['price_bin'] = pd.cut(df['price'], bins=price_bins, labels=price_labels)
print(df['price_bin'].value_counts())

In [ ]:
print(df.describe())

In [ ]:
print('Mean price:', df['price'].mean())
print('Median price:', df['price'].median())
print('Std price:', df['price'].std())

In [ ]:
print('Skewness:\n', df.skew(numeric_only=True))

In [ ]:
avg_price_bed = df.groupby('bedrooms')['price'].mean()
print(avg_price_bed)

In [ ]:
avg_area = df.groupby('bedrooms')['sqft'].mean()
print(avg_area)

In [ ]:
grouped = df.groupby('bedrooms')['price'].agg(['mean', 'min', 'max'])
print(grouped)

In [ ]:
df.to_csv("/tmp/portland_housing_clean.csv", index=False)
print("Preprocessed data saved to /tmp/portland_housing_clean.csv")

In [ ]:
print("Final dataset shape:", df.shape)
print("Columns:", df.columns.tolist())